# FM Vision Encoder Comparison — RadHarmony showcase

Benchmarks **11 foundation-model image encoders** × **6 probing strategies** × **2 chest-X-ray datasets** (VinDr-CXR + TAIX-Ray).

| Aspect | VinDr-CXR | TAIX-Ray |
|---|---|---|
| Origin | Vietnamese, public | German ICU (Aachen), ~200k bedside CXRs from ~50k patients |
| Split | Official train + test (fixed-split) | Single pool — 5-fold CV |
| Variance | `n_seeds=5` × `n_bootstrap=200` | `n_folds=5` |
| Train pool | full train | capped to `n_train_samples=[50000]` |

Both datasets are **clean** w.r.t. all 11 backbones' pretraining sets (CheXpert / PadChest dropped because of leakage).

**Backbones**:
- Vision-only (6): RAD-DINO, DINOv3, CheXFound, Ark+, Medical-MAE, EVA-X
- Vision-language (5, zero-shot capable): BiomedCLIP, CheXagent, MedSigLIP, MedImageInsights, SigLIP2

**Probes**: linear / kNN / SVM / prototype / zero-shot (VL only) / finetune (10 epochs).

**Verification workflow**: shrink `BACKBONES_TO_RUN` / `PROBES_TO_RUN` below to one entry each, step through, confirm CSV schemas. Then expand to the full grid.


## Imports


In [ ]:
import os

# Control which GPU(s) are visible to this notebook.
# Set CUDA_VISIBLE_DEVICES in your shell before launching Jupyter to override,
# e.g.:  CUDA_VISIBLE_DEVICES=2 jupyter lab
# Or change the default value below.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "7")
print(f"CUDA_VISIBLE_DEVICES = {os.environ['CUDA_VISIBLE_DEVICES']}")

In [ ]:
from __future__ import annotations

import gc
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from radharmony.evaluator import (
    LinearProbeEvaluator,
    KNNProbeEvaluator,
    SVMProbeEvaluator,
    PrototypeProbeEvaluator,
    ZeroShotEvaluator,
    FinetuneEvaluator,
)


## Configuration

Edit the three `*_TO_RUN` lists to control what executes. Verification phase: keep them minimal until the schemas check out.


In [ ]:
OUTPUT_DIR  = Path("./eval_out/fm_comparison/")
RESULTS_DIR = OUTPUT_DIR / "results"
# Same convention as notebooks/backbones/done/*: per-(backbone × dataset) subdir under one cache root.
# Both the MONAI PersistentDataset dir and the embedding-cache .pkl live here.
CACHE_ROOT  = Path("/path/to/cache/")
for d in (OUTPUT_DIR, RESULTS_DIR, CACHE_ROOT):
    d.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda:0"

# ─── what to run ───────────────────────────────────────────────────────
BACKBONES_TO_RUN =  [
    # vision-only
    "rad_dino", "dinov3", "chexfound", "ark_plus", "medical_mae", "eva_x",
    # vision-language
    "biomed_clip", "chexagent", "medsiglip", "medimageinsights", "siglip2",
]
PROBES_TO_RUN    = ["linear", "knn", "svm", "prototype", "zero_shot", "finetune"]  
DATASETS_TO_RUN  = ["vindr_cxr", "taix_ray"]

# Per-dataset label subset. None = use every LABEL_COLS the dataset exposes.
# To restrict, set the value to a list of label names (matched case-insensitively
# against the dataset's LABEL_COLS). The evaluator's `labels=` kwarg propagates
# through every probe — results.csv will only contain rows for the filtered labels.
# Note: existing results.csv files are NOT regenerated when you change the filter
# (the resume-safe check is per-directory). Delete the relevant subdirs under
# RESULTS_DIR / <dataset>/<probe>/<backbone>/ to force a refit.
# Examples:
#   {"vindr_cxr": ["pneumothorax", "cardiomegaly"], "taix_ray": None}
#   {"vindr_cxr": None, "taix_ray": ["heart_size"]}
LABELS_TO_RUN: dict[str, list[str] | None] = {
    "vindr_cxr": ['aortic_enlargement',
 'atelectasis',
 'calcification',
 'cardiomegaly',
 'consolidation',
 'ild',
 'infiltration',
 'lung_opacity',
 'lung_tumor',
 'no_finding',
 'nodule/mass',
 'other_diseases',
 'other_lesion',
 'pleural_effusion',
 'pleural_thickening',
 'pneumonia',
 'pulmonary_fibrosis',
 'tuberculosis'],
    "taix_ray":  None,
}
# ───────────────────────────────────────────────────────────────────────

# VinDr fixed-split protocol
VINDR_N_SEEDS     = 5
VINDR_N_BOOTSTRAP = 200

# TAIX-Ray 5-fold protocol
TAIX_N_FOLDS      = 5
TAIX_N_TRAIN      = 50_000

# Common
BASE_SEED         = 0
THRESHOLD         = "youden"
BATCH_SIZE        = 16
NUM_WORKERS       = 4
AUTOCAST_DTYPE    = torch.bfloat16

# Finetune
FT_EPOCHS              = 20
FT_LR                  = 1e-4
# Early stopping for FinetuneEvaluator — best-val-metric tracking + patience-based exit.
# Both k-fold and fixed-split modes carve a FT_VAL_FRACTION slice from the train
# subsample for the val signal; the held-out fold / test set is used only for scoring.
# Set FT_VAL_FRACTION=0.0 to disable early stopping.
FT_EARLY_STOP_METRIC   = "auroc"   # any key in the cls metric panel: auroc|auprc|f1|accuracy|...
FT_EARLY_STOP_PATIENCE = 5
FT_VAL_FRACTION        = 0.1


## Backbone registry

Each entry is `(factory_callable, factory_kwargs, has_text_encoder)`.
`has_text_encoder=True` factories return the 4-tuple
`(transform, image_encoder, text_encoder, processor)`; vision-only factories
return `(transform, image_encoder)`.


In [ ]:
from radharmony.evaluator.backbones import (
    make_raddino, make_dinov3, make_chexfound, make_ark_plus,
    make_medical_mae, make_eva_x,
    make_biomed_clip, make_chexagent, make_medsiglip,
    make_medimageinsights, make_siglip2,
)

BACKBONE_FACTORIES: dict = {
    # vision-only
    "rad_dino":         (make_raddino,         {"device": DEVICE}, False),
    "dinov3":           (make_dinov3,          {"device": DEVICE}, False),
    "chexfound":        (make_chexfound,       {"device": DEVICE}, False),
    "ark_plus":         (make_ark_plus,        {"device": DEVICE}, False),
    "medical_mae":      (make_medical_mae,     {"device": DEVICE}, False),
    "eva_x":            (make_eva_x,           {"device": DEVICE}, False),
    # vision-language
    "biomed_clip":      (make_biomed_clip,     {"device": DEVICE}, True),
    "chexagent":        (make_chexagent,       {"device": DEVICE}, True),
    "medsiglip":        (make_medsiglip,       {"device": DEVICE}, True),
    "medimageinsights": (make_medimageinsights,{"device": DEVICE}, True),
    "siglip2":          (make_siglip2,         {"device": DEVICE}, True),}


## Dataset builders

Each builder takes a backbone-specific `transform` and returns the
RadHarmony dataset. NAS paths are inlined; edit for your environment.


In [ ]:
from radharmony.dataset import (
    VinDrCXRTrainDataset,
    VinDrCXRTestDataset,
    TAIXRay512Dataset,
)

VINDR_TRAIN_DIR = "/path/to/VinDr-CXR/physionet.org/files/vindr-cxr/1.0.0/train/"
VINDR_TEST_DIR  = "/path/to/VinDr-CXR/physionet.org/files/vindr-cxr/1.0.0/test/"
TAIX_IMG_DIR    = "/path/to/TAIX-Ray/data_512/images/"

def build_vindr_train(transform, cache_dir=None):
    return VinDrCXRTrainDataset(
        base_image_dir=VINDR_TRAIN_DIR, transform=transform,
        cache_dir=cache_dir, output_cls=True,
    )

def build_vindr_test(transform, cache_dir=None):
    return VinDrCXRTestDataset(
        base_image_dir=VINDR_TEST_DIR, transform=transform,
        cache_dir=cache_dir, output_cls=True,
    )

def build_taix_ray(transform, cache_dir=None):
    return TAIXRay512Dataset(
        base_image_dir=TAIX_IMG_DIR, transform=transform,
        cache_dir=cache_dir, output_cls=True,
    )


## Zero-shot prompts

Per-label ensemble of 3 templates, restricted to whatever `LABEL_COLS` the
current dataset exposes. `_` is replaced with space to read more naturally
through the text encoder.


In [ ]:
def make_prompts(labels):
    return {
        lbl: [
            f"a chest X-ray showing {lbl.replace('_', ' ')}",
            f"radiograph with {lbl.replace('_', ' ')}",
            f"{lbl.replace('_', ' ')} present on chest X-ray",
        ]
        for lbl in labels
    }


def make_negative_prompts(labels):
    return {
        lbl: [
            f"a chest X-ray with no {lbl.replace('_', ' ')}",
            f"radiograph showing no {lbl.replace('_', ' ')}",
            f"normal chest X-ray without {lbl.replace('_', ' ')}",
        ]
        for lbl in labels
    }


## Probe runners

`run_vindr` issues a **fixed-split** call (train + test, seeds × bootstrap).
`run_taix` issues a **k-fold** call with the 50 k train cap. Both share the
same `embedding_cache` per (backbone × dataset) so the 5 non-finetune probes
extract embeddings exactly once.


In [ ]:
def _has_result(dataset_key, probe_key, backbone_key):
    return (RESULTS_DIR / dataset_key / probe_key / backbone_key / "results.csv").exists()


def _output_dir(dataset_key, probe_key, backbone_key):
    p = RESULTS_DIR / dataset_key / probe_key / backbone_key
    p.mkdir(parents=True, exist_ok=True)
    return str(p)


def _embed_cache(backbone_key, dataset_key):
    # evaluator appends '.train.pkl' / '.test.pkl' automatically in fixed-split mode
    return str(CACHE_ROOT / f"{backbone_key}_{dataset_key}")


def _monai_cache(backbone_key, dataset_split):
    # MONAI PersistentDataset cache — one dir per (backbone × dataset × split)
    return str(CACHE_ROOT / f"{backbone_key}_{dataset_split}")


In [ ]:
def run_vindr(probe_key, image_encoder, train_ds, test_ds, backbone_key, *,
              text_encoder=None, tokenizer=None):
    vindr_labels = LABELS_TO_RUN.get("vindr_cxr")
    common = dict(
        train_dataset=train_ds,
        test_dataset=test_ds,
        labels=vindr_labels,
        device=DEVICE,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        autocast_dtype=AUTOCAST_DTYPE,
        embedding_cache=_embed_cache(backbone_key, "vindr_cxr"),
        output_dir=_output_dir("vindr_cxr", probe_key, backbone_key),
        base_seed=BASE_SEED,
        threshold_strategy=THRESHOLD,
    )
    if probe_key == "linear":
        ev = LinearProbeEvaluator(image_encoder, l2_normalize=True,
            n_seeds=VINDR_N_SEEDS, n_bootstrap=VINDR_N_BOOTSTRAP, **common)
    elif probe_key == "knn":
        ev = KNNProbeEvaluator(image_encoder, k=20,
            n_seeds=VINDR_N_SEEDS, n_bootstrap=VINDR_N_BOOTSTRAP, **common)
    elif probe_key == "svm":
        ev = SVMProbeEvaluator(image_encoder,
            n_seeds=VINDR_N_SEEDS, n_bootstrap=VINDR_N_BOOTSTRAP, **common)
    elif probe_key == "prototype":
        ev = PrototypeProbeEvaluator(image_encoder,
            n_seeds=VINDR_N_SEEDS, n_bootstrap=VINDR_N_BOOTSTRAP, **common)
    elif probe_key == "zero_shot":
        prompt_labels = vindr_labels if vindr_labels is not None else list(test_ds.LABEL_COLS)
        ev = ZeroShotEvaluator(
            image_encoder=image_encoder,
            text_encoder=text_encoder,
            tokenizer=tokenizer,
            prompts=make_prompts(prompt_labels),
            negative_prompts=make_negative_prompts(prompt_labels),
            test_dataset=test_ds,
            labels=vindr_labels,
            device=DEVICE,
            batch_size=BATCH_SIZE,
            num_workers=NUM_WORKERS,
            autocast_dtype=AUTOCAST_DTYPE,
            embedding_cache=_embed_cache(backbone_key, "vindr_cxr"),
            output_dir=_output_dir("vindr_cxr", probe_key, backbone_key),
            base_seed=BASE_SEED,
            threshold_strategy=THRESHOLD,
            n_bootstrap=VINDR_N_BOOTSTRAP,
        )
    elif probe_key == "finetune":
        ev = FinetuneEvaluator(image_encoder,
            epochs=FT_EPOCHS, lr=FT_LR,
            early_stop_metric=FT_EARLY_STOP_METRIC,
            early_stop_patience=FT_EARLY_STOP_PATIENCE,
            val_fraction=FT_VAL_FRACTION,
            n_seeds=VINDR_N_SEEDS, n_bootstrap=VINDR_N_BOOTSTRAP, **common)
    else:
        raise ValueError(f"unknown probe: {probe_key}")

    df = ev.evaluate()
    ev.save_results(df)
    return df


In [ ]:
def run_taix(probe_key, image_encoder, pool_ds, backbone_key, *,
             text_encoder=None, tokenizer=None):
    taix_labels = LABELS_TO_RUN.get("taix_ray")
    common = dict(
        dataset=pool_ds,
        labels=taix_labels,
        device=DEVICE,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        autocast_dtype=AUTOCAST_DTYPE,
        embedding_cache=_embed_cache(backbone_key, "taix_ray"),
        output_dir=_output_dir("taix_ray", probe_key, backbone_key),
        base_seed=BASE_SEED,
        threshold_strategy=THRESHOLD,
    )
    if probe_key == "linear":
        ev = LinearProbeEvaluator(image_encoder, l2_normalize=True,
            n_folds=TAIX_N_FOLDS, n_train_samples=[TAIX_N_TRAIN], **common)
    elif probe_key == "knn":
        ev = KNNProbeEvaluator(image_encoder, k=20,
            n_folds=TAIX_N_FOLDS, n_train_samples=[TAIX_N_TRAIN], **common)
    elif probe_key == "svm":
        ev = SVMProbeEvaluator(image_encoder,
            n_folds=TAIX_N_FOLDS, n_train_samples=[TAIX_N_TRAIN], **common)
    elif probe_key == "prototype":
        ev = PrototypeProbeEvaluator(image_encoder,
            n_folds=TAIX_N_FOLDS, n_train_samples=[TAIX_N_TRAIN], **common)
    elif probe_key == "zero_shot":
        # zero-shot has no fit step: use the full pool as the test set, no folds
        prompt_labels = taix_labels if taix_labels is not None else list(pool_ds.LABEL_COLS)
        ev = ZeroShotEvaluator(
            image_encoder=image_encoder,
            text_encoder=text_encoder,
            tokenizer=tokenizer,
            prompts=make_prompts(prompt_labels),
            negative_prompts=make_negative_prompts(prompt_labels),
            test_dataset=pool_ds,
            labels=taix_labels,
            device=DEVICE,
            batch_size=BATCH_SIZE,
            num_workers=NUM_WORKERS,
            autocast_dtype=AUTOCAST_DTYPE,
            embedding_cache=_embed_cache(backbone_key, "taix_ray"),
            output_dir=_output_dir("taix_ray", probe_key, backbone_key),
            base_seed=BASE_SEED,
            threshold_strategy=THRESHOLD,
            n_bootstrap=200,
        )
    elif probe_key == "finetune":
        ev = FinetuneEvaluator(image_encoder,
            epochs=FT_EPOCHS, lr=FT_LR,
            early_stop_metric=FT_EARLY_STOP_METRIC,
            early_stop_patience=FT_EARLY_STOP_PATIENCE,
            val_fraction=FT_VAL_FRACTION,
            n_folds=TAIX_N_FOLDS, n_train_samples=[TAIX_N_TRAIN], **common)
    else:
        raise ValueError(f"unknown probe: {probe_key}")

    df = ev.evaluate()
    ev.save_results(df)
    return df


## Main loop — per backbone

For each backbone in `BACKBONES_TO_RUN`:
1. Build encoder + transform via factory.
2. Build VinDr (train, test) and TAIX-Ray (pool) datasets with that transform.
3. Run each probe in `PROBES_TO_RUN` on both datasets where applicable.
4. Tear down the encoder and clear CUDA cache before the next backbone.

Per (dataset × probe × backbone) is resume-safe — if `results.csv` exists,
the cell skips it.


In [ ]:
for backbone_key in BACKBONES_TO_RUN:
    factory, kwargs, has_text = BACKBONE_FACTORIES[backbone_key]
    print(f"\n[backbone] {backbone_key}")

    if has_text:
        transform, encoder, text_encoder, tokenizer = factory(**kwargs)
        extra = {"text_encoder": text_encoder, "tokenizer": tokenizer}
    else:
        transform, encoder = factory(**kwargs)
        text_encoder = None
        extra = {}

    train_ds = (build_vindr_train(transform, cache_dir=_monai_cache(backbone_key, "vindr_cxr_train"))
                if "vindr_cxr" in DATASETS_TO_RUN else None)
    test_ds  = (build_vindr_test(transform,  cache_dir=_monai_cache(backbone_key, "vindr_cxr_test"))
                if "vindr_cxr" in DATASETS_TO_RUN else None)
    pool_ds  = (build_taix_ray(transform,    cache_dir=_monai_cache(backbone_key, "taix_ray"))
                if "taix_ray"  in DATASETS_TO_RUN else None)

    for probe_key in PROBES_TO_RUN:
        if probe_key == "zero_shot" and not has_text:
            print(f"  skip {probe_key}: vision-only backbone")
            continue

        if "vindr_cxr" in DATASETS_TO_RUN:
            if _has_result("vindr_cxr", probe_key, backbone_key):
                print(f"  skip vindr_cxr/{probe_key}: results.csv exists")
            else:
                print(f"  running vindr_cxr/{probe_key} ...")
                run_vindr(probe_key, encoder, train_ds, test_ds, backbone_key, **extra)

        if "taix_ray" in DATASETS_TO_RUN:
            if _has_result("taix_ray", probe_key, backbone_key):
                print(f"  skip taix_ray/{probe_key}: results.csv exists")
            else:
                print(f"  running taix_ray/{probe_key} ...")
                run_taix(probe_key, encoder, pool_ds, backbone_key, **extra)

    # release GPU before next backbone
    del encoder
    if text_encoder is not None:
        del text_encoder
    gc.collect()
    torch.cuda.empty_cache()


## Aggregate

Walk `RESULTS_DIR`, concatenate every `results.csv`, tag rows with their
`{dataset, evaluator, backbone}`, and write `all_results.csv`.


In [ ]:
records = []
for csv in RESULTS_DIR.glob("*/*/*/results.csv"):
    parts = csv.relative_to(RESULTS_DIR).parts  # (dataset, evaluator, backbone, "results.csv")
    df = pd.read_csv(csv)
    df["dataset"]   = parts[0]
    df["evaluator"] = parts[1]
    df["backbone"]  = parts[2]
    records.append(df)

all_df = pd.concat(records, ignore_index=True) if records else pd.DataFrame()
all_df.to_csv(OUTPUT_DIR / "all_results.csv", index=False)
print(f"Wrote {len(all_df):,} rows to {OUTPUT_DIR / 'all_results.csv'}")
all_df.head()
